# 02 - ARIMA
**Entrega 2.** Modelo ARIMA con seleccion automatica de orden por indice.

In [5]:
import sys, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')
from utils import load_data, compute_rmse, make_submission, train_val_split, INDEX_COLS
data = load_data()
train_full = data['train_indices'][INDEX_COLS]
test_dates = data['test_dates'].index
train, val = train_val_split(train_full, val_size=252)


## Ajuste ARIMA por indice

In [6]:
def fit_arima(series):
    try:
        import pmdarima as pm
        m = pm.auto_arima(series, stepwise=True, suppress_warnings=True,
                          error_action='ignore', max_p=5, max_q=5, max_d=2)
        return m, 'pmdarima'
    except ImportError:
        from statsmodels.tsa.arima.model import ARIMA
        return ARIMA(series, order=(1,1,1)).fit(), 'statsmodels'

def forecast_arima(model, n, lib):
    if lib == 'pmdarima': return model.predict(n_periods=n)
    return model.forecast(steps=n).values

def arima_all(train, dates, verbose=True):
    preds = {}
    for col in INDEX_COLS:
        if verbose: print(f'  {col} ...', end=' ', flush=True)
        m, lib = fit_arima(train[col])
        preds[col] = forecast_arima(m, len(dates), lib)
        if verbose: print(f'({lib})')
    return pd.DataFrame(preds, index=dates)


## Validacion local

In [7]:
print('Ajustando ARIMA en train...')
pred_val = arima_all(train, val.index)
rmse = compute_rmse(val, pred_val)
print('[ARIMA] RMSE =', round(rmse, 2))
per = np.sqrt(((val.values - pred_val.values)**2).mean(axis=0))
for col, r in zip(INDEX_COLS, per): print(f'  {col}: {round(r,2)}')


Ajustando ARIMA en train...
  Index_A ... (statsmodels)
  Index_B ... (statsmodels)
  Index_C ... (statsmodels)
  Index_D ... (statsmodels)
  Index_E ... (statsmodels)
  Index_F ... (statsmodels)
[ARIMA] RMSE = 3267.44
  Index_A: 3237.47
  Index_B: 398.75
  Index_C: 0.83
  Index_D: 3507.31
  Index_E: 9.91
  Index_F: 12450.37


## Generar submission

In [8]:
print('Prediciendo test...')
pred_test = arima_all(train_full, test_dates)
path = make_submission(pred_test, 'submission_02_arima.csv')
pred_test.head()


Prediciendo test...
  Index_A ... (statsmodels)
  Index_B ... (statsmodels)
  Index_C ... (statsmodels)
  Index_D ... (statsmodels)
  Index_E ... (statsmodels)
  Index_F ... (statsmodels)
Submission saved: c:\Users\1jose\Desktop\previa hackatlon\hackathon\submissions\submission_02_arima.csv


,Index_A,Index_B,Index_C,Index_D,Index_E,Index_F
Date,,,,,,
2024-01-01,16832.158956,4771.137551,20.237302,16840.029410,129.016718,42120.731633
2024-01-02,16830.149663,4770.610857,20.238195,16839.835942,129.010104,42141.223201
2024-01-03,16830.797774,4770.823027,20.238015,16839.854859,129.012952,42160.909949
2024-01-04,16830.588721,4770.737558,20.238051,16839.853009,129.011726,42179.823487
2024-01-05,16830.656153,4770.771988,20.238044,16839.853190,129.012254,42197.994182
